# 다양한 Chat Model 활용


> 업데이트 기준: **2026-09-18**  
> 책의 학습 목표는 유지하면서 LangChain 1.x의 분리된 provider 패키지와 현재 메시지/스트리밍 API에 맞췄습니다. 모델 이름은 공급자 정책에 따라 바뀔 수 있으므로 환경 변수로 덮어쓸 수 있게 구성했습니다.


## 설치와 환경 변수

통합별 패키지는 `langchain` 본체와 독립적으로 배포됩니다. 사용하는 공급자 패키지만 설치해도 됩니다.

`.env`에는 필요한 키만 저장합니다: `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `PPLX_API_KEY`, `TOGETHER_API_KEY`, `COHERE_API_KEY`, `UPSTAGE_API_KEY`. LangSmith 추적은 `LANGSMITH_API_KEY`가 있을 때만 켭니다.


In [ ]:
%pip install -qU langchain python-dotenv langchain-openai langchain-anthropic langchain-perplexity langchain-together langchain-cohere langchain-upstage


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if os.getenv("LANGSMITH_API_KEY"):
    os.environ.setdefault("LANGSMITH_TRACING", "true")
    os.environ.setdefault("LANGSMITH_PROJECT", "CH04-Models-modernized")


def print_stream(chunks):
    # AIMessageChunk를 공급자와 무관하게 텍스트로 출력합니다.
    for chunk in chunks:
        print(chunk.text, end="", flush=True)
    print()


## 공통 초기화 API

공급자를 런타임에 바꾸려면 `init_chat_model()`이 편리합니다. `provider:model` 형식은 애플리케이션 코드를 공급자별 클래스에서 분리합니다. 공급자 고유 옵션을 깊게 사용할 때는 아래의 전용 클래스를 사용합니다.


In [ ]:
from langchain.chat_models import init_chat_model

openai_model = os.getenv("OPENAI_MODEL", "gpt-5-mini")
gpt = init_chat_model(f"openai:{openai_model}", temperature=0)

print_stream(gpt.stream("사랑이 무엇인지 두 문장으로 설명해 주세요."))


## Anthropic

모델 목록과 정확한 ID는 [Anthropic 모델 문서](https://docs.anthropic.com/en/docs/about-claude/models)에서 확인합니다. 날짜가 고정된 ID 대신 환경 변수로 교체 가능한 기본값을 사용합니다.


In [ ]:
from langchain_anthropic import ChatAnthropic

anthropic = ChatAnthropic(
    model=os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6"),
    temperature=0,
    max_tokens=512,
)
print_stream(anthropic.stream("사랑이 무엇인지 두 문장으로 설명해 주세요."))


## Perplexity

커뮤니티 보조 클래스 대신 공식 LangChain 통합 패키지 `langchain-perplexity`를 사용합니다. 인용과 검색 결과는 `AIMessage.additional_kwargs`에 저장됩니다.


In [ ]:
from langchain_perplexity import ChatPerplexity

perplexity = ChatPerplexity(
    model=os.getenv("PERPLEXITY_MODEL", "sonar"),
    temperature=0,
)

response = perplexity.invoke("최근 발표된 LangChain의 주요 변경점을 요약해 주세요.")
print(response.text)

print("\n[인용]")
for index, url in enumerate(response.additional_kwargs.get("citations", []), start=1):
    print(f"{index}. {url}")

print("\n[검색 결과 일부]")
for item in response.additional_kwargs.get("search_results", [])[:3]:
    print(f"- {item.get('title')}: {item.get('url')}")


In [ ]:
# 스트리밍 청크를 합치면 마지막 메타데이터까지 안정적으로 확인할 수 있습니다.
full_response = None
for chunk in perplexity.stream("LangChain의 최신 릴리스 노트를 세 문장으로 요약해 주세요."):
    print(chunk.text, end="", flush=True)
    full_response = chunk if full_response is None else full_response + chunk

print("\n\n[인용]")
for index, url in enumerate(
    full_response.additional_kwargs.get("citations", []), start=1
):
    print(f"{index}. {url}")


## Together AI

구조화 출력은 문자열 JSON을 직접 파싱하지 않고 Pydantic 스키마로 검증합니다.


In [ ]:
from pydantic import BaseModel, Field
from langchain_together import ChatTogether


class Lotto(BaseModel):
    numbers: list[int] = Field(description="1부터 45까지의 중복 없는 숫자 6개")


together = ChatTogether(
    model=os.getenv(
        "TOGETHER_MODEL", "meta-llama/Llama-3.3-70B-Instruct-Turbo"
    ),
    temperature=0,
)
lotto_model = together.with_structured_output(Lotto)
result = lotto_model.invoke("로또 번호 6개를 추천해 주세요.")
print(result)


## Cohere와 Upstage


In [ ]:
from langchain_cohere import ChatCohere
from langchain_upstage import ChatUpstage

cohere = ChatCohere(
    model=os.getenv("COHERE_MODEL", "command-a-03-2025"),
    temperature=0,
)
print_stream(cohere.stream("RAG가 필요한 이유를 두 문장으로 설명해 주세요."))

# 모델을 생략하면 설치된 통합 패키지의 현재 기본 채팅 모델을 사용합니다.
upstage = ChatUpstage(temperature=0)
print_stream(upstage.stream("RAG가 필요한 이유를 두 문장으로 설명해 주세요."))


## OpenAI 호환 엔드포인트

특정 서비스의 만료된 URL·모델·키를 코드에 박아 두지 않습니다. OpenAI 호환 API라면 아래 세 환경 변수로 연결합니다. 키는 노트북이나 Git 저장소에 기록하지 않습니다.


In [ ]:
from langchain_openai import ChatOpenAI

required = [
    "OPENAI_COMPATIBLE_BASE_URL",
    "OPENAI_COMPATIBLE_API_KEY",
    "OPENAI_COMPATIBLE_MODEL",
]
missing = [name for name in required if not os.getenv(name)]

if missing:
    print("실행하려면 환경 변수를 설정하세요:", ", ".join(missing))
else:
    compatible_model = ChatOpenAI(
        base_url=os.environ["OPENAI_COMPATIBLE_BASE_URL"],
        api_key=os.environ["OPENAI_COMPATIBLE_API_KEY"],
        model=os.environ["OPENAI_COMPATIBLE_MODEL"],
        temperature=0,
    )
    print_stream(compatible_model.stream("안녕하세요!"))
